# Notebook 2 — Applying Concentration Inequalities to Quantum Circuits

**Learning goals:**

1. Take the concentration inequalities from Notebook 1
   (Hoeffding, Chernoff, Chebyshev, Markov-as-foil)
   and apply them to *actual quantum circuits*.
2. See concretely how:
      – a quantum observable  `⟨O⟩`
      – becomes measurement outcomes `X_i`
      – becomes a sample mean `p_hat`
      – gets a confidence interval from concentration inequalities.
3. Understand how IBM's **NEAT** (`qiskit_ibm_runtime.debug_tools.Neat`)
   can give us a high-quality "ground truth" expectation value
   for comparison against finite-shot Estimator runs.

Structure:

1. Local, hardware-free example using Qiskit SDK primitives (`StatevectorEstimator`)
   to get "ideal" expectation values and then simulate finite shots.
2. Conceptual + code template: How to integrate the *same logic*
   with Qiskit Runtime `EstimatorV2` + `Neat.ideal_sim(...)` on IBM Quantum.

## 0. Setup: Imports & Concentration Inequalities

We reuse a subset of the helpers from Notebook 1:
  - hoeffding_bound, chernoff_bound, chebyshev_bound
  - sample_size_hoeffding, sample_size_chebyshev, sample_size_chernoff

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(123)

# --- Concentration bounds on Bernoulli means ---

def hoeffding_bound(p_est: float, n: int, delta: float):
    """Two-sided Hoeffding CI for Bernoulli / [0,1]-bounded variables."""
    if n <= 0:
        raise ValueError("n must be positive")

    radius = math.sqrt(math.log(2 / delta) / (2 * n))
    return max(0.0, p_est - radius), min(1.0, p_est + radius)


def chernoff_bound(p_est: float, n: int, delta: float):
    """Simple Chernoff-style CI: P(|p_hat - p| >= eps) ≤ 2 exp(- n eps^2 / 3)."""
    if n <= 0:
        raise ValueError("n must be positive")

    radius = math.sqrt(3 * math.log(2 / delta) / n)
    return max(0.0, p_est - radius), min(1.0, p_est + radius)


def chebyshev_bound(p_est: float, n: int, delta: float):
    """
    Two-sided Chebyshev CI using *empirical* variance for Bernoulli.
    Not strictly guaranteed, but useful as a contrast.
    """
    if n <= 0:
        raise ValueError("n must be positive")

    var_est = p_est * (1 - p_est)
    if var_est == 0:
        return p_est, p_est

    radius = math.sqrt(var_est / (delta * n))
    return max(0.0, p_est - radius), min(1.0, p_est + radius)


def sample_size_hoeffding(epsilon: float, delta: float) -> int:
    """Solve 2 exp(-2 n ε²) ≤ δ for n."""
    if epsilon <= 0 or delta <= 0 or delta >= 1:
        raise ValueError("epsilon > 0 and 0 < delta < 1 required.")
    return int(math.ceil((1.0 / (2 * epsilon**2)) * math.log(2.0 / delta)))


def sample_size_chebyshev(epsilon: float, delta: float) -> int:
    """Chebyshev with worst-case variance σ² ≤ 1/4."""
    if epsilon <= 0 or delta <= 0 or delta >= 1:
        raise ValueError("epsilon > 0 and 0 < delta < 1 required.")
    return int(math.ceil(1.0 / (4 * epsilon**2 * delta)))


def sample_size_chernoff(epsilon: float, delta: float) -> int:
    """Chernoff-style bound sample complexity."""
    if epsilon <= 0 or delta <= 0 or delta >= 1:
        raise ValueError("epsilon > 0 and 0 < delta < 1 required.")
    return int(math.ceil((3.0 / (epsilon**2)) * math.log(2.0 / delta)))


## 1. A Minimal Quantum Example: 1-Qubit RY + Z Observable

We build a 1-qubit circuit:

```
  |0> -- RY(θ) --> cos(θ/2)|0> + sin(θ/2)|1>
```

If we measure `Z` (eigenvalues `+1` for `|0>`, `-1` for `|1>`), the ideal expectation is:

```
  E[Z] = cos^2(θ/2) - sin^2(θ/2) = cos(θ).
```

If instead we think in terms of a Bernoulli variable:

```
  X ∈ {0,1}, with X = 1 if we get outcome "0",
```

then:

```
  P(X=1) = P(0) = cos^2(θ/2) = (1 + cos θ) / 2.
```

That `"P(0)"` is exactly the type of quantity that concentration inequalities in Notebook 1 are about.


In [2]:

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def one_qubit_ry_circuit(theta: float) -> QuantumCircuit:
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    return qc

def analytic_p0(theta: float) -> float:
    """Analytic P(measure 0) for |ψ> = RY(θ)|0>."""
    return math.cos(theta / 2) ** 2

# Quick sanity check:
for th in [0, math.pi/3, math.pi/2, math.pi]:
    print(f"theta = {th:.3f}, analytic P(0) = {analytic_p0(th):.4f}")


ModuleNotFoundError: No module named 'qiskit'

### 1.1 Numerical ideal P(0) via Statevector

To mimic what NEAT's `ideal_sim` does conceptually, we use an exact simulator.
For each `θ`, we:
  - build the circuit,
  - get the statevector,
  - compute `P(0)` from the amplitudes,
  - compare to the analytic value.

In [ ]:

def statevector_p0(theta: float) -> float:
    qc = one_qubit_ry_circuit(theta)
    sv = Statevector.from_instruction(qc)
    # Probability of measuring |0> on qubit 0:
    # For 1 qubit, amps[0] is |0>, amps[1] is |1>
    probs = sv.probabilities()
    return float(probs[0])

thetas = np.linspace(0, math.pi, 6)
for th in thetas:
    p_analytic = analytic_p0(th)
    p_sv = statevector_p0(th)
    print(f"θ={th:.3f}  analytic={p_analytic:.4f},  statevector={p_sv:.4f}")


## 2. From Ideal P(0) to Finite-Shot Estimation

Now we:
1. Choose a target accuracy (`ε`) and confidence level (`1 - δ`).
2. Use Hoeffding to compute a **shot budget m**.
3. For several different `θ`, we:
   - get `p0_ideal`,
   - simulate `m` shots,
   - compute `p0_hat`,
   - compute `|p0_hat - p0_ideal|`,
   - check whether it exceeds `ε`.


In [ ]:
epsilon = 0.05
delta = 0.05
m_hoeff = sample_size_hoeffding(epsilon, delta)
print(f"Hoeffding shot budget m = {m_hoeff} for ε = {epsilon}, δ = {delta}")

def simulate_shots_p0(theta: float, shots: int, rng: np.random.Generator) -> float:
    p0 = statevector_p0(theta)  # ideal underlying Bernoulli parameter
    # We treat the measurement as Bernoulli(p0) and sample 'shots' times:
    counts_0 = rng.binomial(n=shots, p=p0)
    return counts_0 / shots

results = []
for idx, th in enumerate(np.linspace(0, math.pi, 10), start=1):
    p0_ideal = statevector_p0(th)
    p0_hat = simulate_shots_p0(th, m_hoeff, rng)
    err = abs(p0_hat - p0_ideal)
    results.append((idx, th, p0_ideal, p0_hat, err))

print("Using Hoeffding shot budget")
for idx, th, p0_ideal, p0_hat, err in results:
    print(f"Case {idx:02d}: θ={th:.3f}, p0_ideal={p0_ideal:.4f}, p0_hat={p0_hat:.4f}, |err|={err:.4f}")

violations = sum(1 for _,_,_,_,err in results if err > epsilon)
print("\n=== Summary ===")
print(f"Total cases              : {len(results)}")
print(f"Violations (> ε={epsilon}) : {violations}")
print(f"Empirical violation rate : {violations / len(results):.3f} (target δ = {delta})")
print(f"Max abs error            : {max(err for *_, err in results):.4f}")


### 2.1 Visualizing Errors vs Hoeffding Radius

Hoeffding radius:
  ```r_H = sqrt(log(2/δ) / (2m))```.

We plot the absolute errors `|p0_hat - p0_ideal|` for each case, and draw
a horizontal line at `r_H` and at `ε`.

In [ ]:

# Compute Hoeffding radius directly for m_hoeff, δ:
hoeff_radius = math.sqrt(math.log(2 / delta) / (2 * m_hoeff))

case_ids = [idx for idx,_,_,_,_ in results]
errors = [err for *_, err in results]

plt.bar(case_ids, errors)
plt.axhline(hoeff_radius, linestyle="--", label=f"Hoeffding radius ≈ {hoeff_radius:.3f}")
plt.axhline(epsilon, linestyle=":", label=f"ε = {epsilon}")
plt.xlabel("Case index")
plt.ylabel("|p0_hat - p0_ideal|")
plt.title("Absolute error per case vs Hoeffding radius")
plt.legend()
plt.show()


## 3. CI Comparison on a Quantum Observable

Here we fix a single θ*, use the statevector to get p0_ideal, then:

- Run many trials:
    * draw m samples (shots),
    * compute p0_hat,
    * build CIs using Hoeffding, Chernoff, Chebyshev,
    * check coverage (whether p0_ideal lies inside),
    * track average width.

This mirrors Notebook 1, but now p0_ideal is genuinely coming from
a quantum circuit.

In [ ]:
def ci_coverage_and_width_for_theta(theta, m, delta, trials=3000):
    p_true = statevector_p0(theta)
    cover_counts = {"Hoeffding": 0, "Chernoff": 0, "Chebyshev": 0}
    width_sums   = {"Hoeffding": 0.0, "Chernoff": 0.0, "Chebyshev": 0.0}

    for _ in range(trials):
        counts_0 = rng.binomial(n=m, p=p_true)
        p_hat = counts_0 / m

        # Hoeffding
        lo_h, hi_h = hoeffding_bound(p_hat, m, delta)
        if lo_h <= p_true <= hi_h:
            cover_counts["Hoeffding"] += 1
        width_sums["Hoeffding"] += (hi_h - lo_h)

        # Chernoff
        lo_c, hi_c = chernoff_bound(p_hat, m, delta)
        if lo_c <= p_true <= hi_c:
            cover_counts["Chernoff"] += 1
        width_sums["Chernoff"] += (hi_c - lo_c)

        # Chebyshev (empirical)
        lo_ch, hi_ch = chebyshev_bound(p_hat, m, delta)
        if lo_ch <= p_true <= hi_ch:
            cover_counts["Chebyshev"] += 1
        width_sums["Chebyshev"] += (hi_ch - lo_ch)

    results = {}
    for name in cover_counts:
        coverage = cover_counts[name] / trials
        avg_width = width_sums[name] / trials
        results[name] = (coverage, avg_width)
    return p_true, results

theta_star = math.pi / 3  # pick a non-trivial angle
m_star = m_hoeff          # reuse Hoeffding shot budget
delta_ci = 0.05

p_true, res = ci_coverage_and_width_for_theta(theta_star, m_star, delta_ci)

print(f"θ* = {theta_star:.3f}, true p0 = {p_true:.4f}, m = {m_star}, δ = {delta_ci}")
print("\nMethod       Coverage   Avg width")
print("---------------------------------")
for name, (cov, width) in res.items():
    print(f"{name:<10} {cov:8.3f}   {width:9.4f}")


## 4. Where NEAT and IBM Runtime Fit In

In the cells above, the "ideal" probability p0_ideal was computed with:

  - an exact statevector simulator (`Statevector.from_instruction`).

For your **real QAMP / thesis pipeline**, you want:

  - *ideal, noiseless* expectation values from **NEAT**:
      - `neat.ideal_sim(pubs, ...)`
  - *noisy, finite-shot* estimates from **EstimatorV2** running on
    real or noisy hardware:
      - `EstimatorV2.run(pubs_with_shots, ...)`

Then you plug the *same* concentration inequalities around the
finite-shot estimates, with:

  - p_hat  := the shot-based estimate (e.g., ancilla Z expectation → p0_hat),
  - p_true := the NEAT ideal_sim expectation, mapped to a Bernoulli parameter
              if needed,
  - m      := shot budget (computed from ε, δ),
  - use Hoeffding/Chernoff/Chebyshev as in this notebook.

Below is a *template* for doing this using:
  - Qiskit Runtime `EstimatorV2`,
  - NEAT from `qiskit_ibm_runtime.debug_tools`.

This requires:
  - IBM Quantum account configured (QiskitRuntimeService),
  - a backend that supports Estimator jobs.


In [ ]:
# %% [markdown]
# ### 4.1 NEAT + EstimatorV2 Template
#
# This is a *pattern*, not something that will run in this offline notebook.
# Use it in an environment where:
#   - you have `qiskit-ibm-runtime` installed,
#   - you are authenticated (`QiskitRuntimeService()` works).

# %%
# This cell is meant as a reference template; comment it out if you
# don't have qiskit-ibm-runtime installed in your environment.

"""
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator
from qiskit_ibm_runtime.debug_tools import Neat

# 1) Connect to the service and pick a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# 2) Build your circuit and observable (here: ancilla Z, like SWAP/Hadamard tests)
qc = QuantumCircuit(1)
theta = 0.7
qc.ry(theta, 0)          # example; in your case, it's SWAP / Hadamard test circuit

# No measurement in Estimator circuits
observable = SparsePauliOp(["Z"], coeffs=[1.0])  # measure Z on qubit 0

# Optional: transpile to backend ISA
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
isa_observable = observable.apply_layout(isa_circuit.layout)

# 3) Pack into a PUB for EstimatorV2
#    A PUB is (circuit, observables, parameter_values, (optional) target_precision)
pubs = [(isa_circuit, isa_observable, [[]])]

# 4) Use NEAT to get ideal, Cliffordized estimates
neat = Neat(backend)
neat_res = neat.ideal_sim(pubs, cliffordize=True)   # ideal, noiseless, Clifford proxy

# NEAT returns NeatResult; get expectation value(s)
p0_ideal_like = neat_res[0].data.evs[0]  # for a Z observable, this is <Z>, not P(0)!

# Map <Z> ∈ [-1,1] to a Bernoulli parameter P(0) ∈ [0,1]
# Recall: <Z> = P(0) - P(1) = 2 P(0) - 1  ⇒  P(0) = ( <Z> + 1 ) / 2
p0_ideal = (p0_ideal_like + 1.0) / 2.0

# 5) Use EstimatorV2 to get finite-shot estimates
estimator = Estimator(mode=backend)

# Choose shot budget via Hoeffding (from Notebook 1 / 2)
epsilon = 0.05
delta = 0.05
m_shots = sample_size_hoeffding(epsilon, delta)

# In EstimatorV2, 'shots' is set via options
job = estimator.run(pubs, options={"shots": m_shots})
pub_result = job.result()[0]
z_hat = pub_result.data.evs[0]      # finite-shot <Z> estimate
p0_hat = (z_hat + 1.0) / 2.0        # convert to Bernoulli parameter

# 6) Apply concentration inequalities around p0_hat
lo_h, hi_h = hoeffding_bound(p0_hat, m_shots, delta)
lo_c, hi_c = chernoff_bound(p0_hat, m_shots, delta)
lo_ch, hi_ch = chebyshev_bound(p0_hat, m_shots, delta)

print(f"Ideal P(0) from NEAT (proxy)    : {p0_ideal:.4f}")
print(f"Finite-shot P(0) from Estimator : {p0_hat:.4f}")
print(f"Hoeffding CI                    : [{lo_h:.4f}, {hi_h:.4f}]")
print(f"Chernoff  CI                    : [{lo_c:.4f}, {hi_c:.4f}]")
print(f"Chebyshev CI                    : [{lo_ch:.4f}, {hi_ch:.4f}]")

# 7) You can repeat for many PUBs (like your 30 pairs),
#    accumulating (p0_ideal, p0_hat, |err|) and violation counts.
"""


## 5. Takeaways

1. **Polls → QKD → Observables**:
   - In Notebook 1 we saw that the same math (Hoeffding, Chernoff, Chebyshev)
     governs error control for polls and QKD parameter estimation.
   - In this notebook we grounded the same math in a quantum observable:
       p0 = P(ancilla = 0).

2. **Ideal vs Finite-shot**:
   - An "ideal" expectation value (from Statevector or NEAT's `ideal_sim`)
     plays the role of the *true mean*.
   - Finite-shot Estimator runs give you a sample mean `p0_hat`.
   - Concentration inequalities tell you, **before running**, how large
     your shot budget needs to be to achieve (ε, δ) guarantees.

3. **NEAT's role**:
   - NEAT `ideal_sim` gives a high-quality Clifford-proxy for your circuits,
     which acts as a "ground truth" to:
       - validate your shot budgeting logic,
       - debug whether observed deviations are due to finite sampling
         or due to hardware noise / drift / modeling mismatch.


